# Exercise: produce_02 — batch producer & partitions

**Goal:** send many events at once and observe how keys map to partitions.

**Prerequisite:** topics `strom` and `wasser` already exist (Redpanda auto-creates them on first write, or run `rpk topic create strom -X brokers=redpanda:29092` once).

In [ ]:
from confluent_kafka import Producer
from collections import defaultdict, Counter
import json, time, random

producer = Producer({'bootstrap.servers': 'redpanda:29092', 'client.id': 'batch-producer'})

houses = ['haus_a', 'haus_b', 'haus_c']
topics = ['strom', 'wasser']

## Step 1 — send 15 random events

Run the cell. **Watch the output:**
- Do events with the *same key* always land in the *same partition*?
- Run the cell a second time — same answer?

In [ ]:
results = []

def delivery_report(err, msg):
    if err:
        print(f'FAILED: {err}')
    else:
        results.append({'topic': msg.topic(), 'partition': msg.partition(), 'key': msg.key().decode()})
        print(f'  -> {msg.topic()} [P{msg.partition()}] key={msg.key().decode()} offset={msg.offset()}')

for _ in range(15):
    house = random.choice(houses)
    topic = random.choice(topics)
    value = round(random.uniform(1.0, 100.0), 2)
    event = json.dumps({
        'sensor':    topic,
        'haus':      house,
        'wert':      value,
        'einheit':   'kWh' if topic == 'strom' else 'Liter',
        'timestamp': time.time(),
    })
    producer.produce(topic, key=house.encode(), value=event.encode(), callback=delivery_report)

producer.flush()
print('15 events sent.')

## Step 2 — analyze the partition mapping

In [ ]:
key_partitions = defaultdict(set)
for r in results:
    key_partitions[(r['topic'], r['key'])].add(r['partition'])

print('Topic   | Key     | Partition(s) | Consistent?')
print('-' * 50)
for (t, k), parts in sorted(key_partitions.items()):
    plist = ', '.join(str(p) for p in sorted(parts))
    flag  = 'yes' if len(parts) == 1 else 'NO!'
    print(f'{t:<7} | {k:<7} | {plist:<12} | {flag}')

## Task A — events without a key

Send 6 events to `strom` **without a key** (`key=None`). Where do they land?

> Expected: round-robin across partitions. **No ordering guarantee** for keyless events.

In [ ]:
no_key_partitions = []

def no_key_cb(err, msg):
    if not err:
        no_key_partitions.append(msg.partition())
        print(f'  -> P{msg.partition()} offset={msg.offset()}')

# TODO: loop 6 times and produce events with key=None and any small JSON value

producer.flush()
print(f'Distribution: {dict(Counter(no_key_partitions))}')

## Task B — verify in the Console

1. Open the **Redpanda Console** → topic `strom` → *Messages*
2. Use the *Partition* filter — pick 0 only, then 1 only.
3. Are the keys you see consistent with the Python output above?